In [1]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [2]:
DATA_PATH = "../data/GamingandMentalHealth_final.csv"


def normalize_train(train):
    """
    normalize_data() : Scales each feature column to 0-1 for training set

    Parameters:
        data : pd df, features

    Returns:
        normalized_data : pd df, normalized features
    """

    norm_data = train.copy()

    # iterate over columns by name
    for col in train.columns:
        f_min = train[col].min()
        f_max = train[col].max()

        if f_max - f_min > 0:
            norm_data[col] = (train[col] - f_min) / (f_max - f_min)
        else:
            norm_data[col] = 0.0

    return norm_data

def normalize_test(train, test):
    """
    normalize_data() : Scales each feature column to 0-1 for test/val sets
                        - using method for both train and test prevents data leakage

    Parameters:
        data : pd df, features

    Returns:
        normalized_data : pd df, normalized features
    """

    norm_data = test.copy()

    # iterate over columns by name
    for col in test.columns:
        f_min = train[col].min()
        f_max = train[col].max()

        if f_max - f_min > 0:
            norm_data[col] = (test[col] - f_min) / (f_max - f_min)
        else:
            norm_data[col] = 0.0

    return norm_data

# MLP IMPLEMENTATION

# load data
data = pd.read_csv(DATA_PATH)

# split target labels from features
X = data.drop(columns="gaming_addiction_risk_level")
y = data["gaming_addiction_risk_level"]

# pull out valiation set
X_full, X_val, y_full, y_val = train_test_split(X, y, test_size=0.2)

# perform train/test split
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=1)

# normalize datasets based on stats in train data
X_train_mm = normalize_train(X_train)
X_test_mm = normalize_test(X_train, X_test)
X_val_mm = normalize_test(X_train, X_val)

In [3]:

# create model
""" 
Model has 1 hidden layer (64 neurons)
layer sizes: (input)25, 64, (output)4
"""
model_1L = MLPClassifier(
    activation='relu',          
    solver='adam',      
    learning_rate_init=0.001,        
    max_iter=1000,              
    alpha=0.001,
    random_state=1,
    hidden_layer_sizes=(64,)    
)


## MLP with one hidden layer
# run train/test loop
model_1L.fit(X_train_mm, y_train)

# test predictions
y_test_pred = model_1L.predict(X_test_mm)

# validation predictions
y_val_pred = model_1L.predict(X_val_mm)

print("VAL (min-max normalization, 1 layer MLP)",
      "\nAccuracy:", accuracy_score(y_val, y_val_pred),
      "\nF1:", f1_score(y_val, y_val_pred, average='weighted'),
      "\nPrecision:", precision_score(y_val, y_val_pred, average='weighted'),
      "\nRecall:", recall_score(y_val, y_val_pred, average='weighted'))

## MLP with multiple hidden layers
# create model
""" 
Model has 2 hidden layers (128, 64 neurons)
layer sizes: (input)25, 128, 64, (output)4
"""
model_2L = MLPClassifier(
    activation='relu',          
    solver='adam',      
    learning_rate_init=0.001,        
    max_iter=1000,              
    alpha=0.001,
    random_state=2,
    hidden_layer_sizes=(128,64,)    
)

## MLP with two hidden layers
# run train/test loop
model_2L.fit(X_train_mm, y_train)

# test predictions
y_test_pred = model_2L.predict(X_test_mm)

# validation predictions
y_val_pred = model_2L.predict(X_val_mm)


print("VAL (min-max normalization, 2 layer MLP)",
      "\nAccuracy:", accuracy_score(y_val, y_val_pred),
      "\nF1:", f1_score(y_val, y_val_pred, average='weighted'),
      "\nPrecision:", precision_score(y_val, y_val_pred, average='weighted'),
      "\nRecall:", recall_score(y_val, y_val_pred, average='weighted'))

VAL (min-max normalization, 1 layer MLP) 
Accuracy: 0.95 
F1: 0.9500246629697493 
Precision: 0.9503701931266071 
Recall: 0.95
VAL (min-max normalization, 2 layer MLP) 
Accuracy: 0.97 
F1: 0.9702034295583519 
Precision: 0.9716258257709101 
Recall: 0.97


In [4]:
# MLP IMPLEMENTATION
# normalize datasets using StandardScalar()
scaler = StandardScaler()

X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)
X_val_std = scaler.transform(X_val)

In [5]:
## MLP with one hidden layer
# run train/test loop
model_1L.fit(X_train_std, y_train)

# test predictions
y_test_pred = model_1L.predict(X_test_std)

# validation predictions
y_val_pred = model_1L.predict(X_val_std)

print("VAL (std normalization, 1 layer MLP)",
      "\nAccuracy:", accuracy_score(y_val, y_val_pred),
      "\nF1:", f1_score(y_val, y_val_pred, average='weighted'),
      "\nPrecision:", precision_score(y_val, y_val_pred, average='weighted'),
      "\nRecall:", recall_score(y_val, y_val_pred, average='weighted'))

## MLP with two hidden layers
# run train/test loop
model_2L.fit(X_train_std, y_train)

# test predictions
y_test_pred = model_2L.predict(X_test_std)

# validation predictions
y_val_pred = model_2L.predict(X_val_std)


print("VAL (std normalization, 2 layer MLP)",
      "\nAccuracy:", accuracy_score(y_val, y_val_pred),
      "\nF1:", f1_score(y_val, y_val_pred, average='weighted'),
      "\nPrecision:", precision_score(y_val, y_val_pred, average='weighted'),
      "\nRecall:", recall_score(y_val, y_val_pred, average='weighted'))


VAL (std normalization, 1 layer MLP) 
Accuracy: 0.94 
F1: 0.9390343349468231 
Precision: 0.9390414453816516 
Recall: 0.94
VAL (std normalization, 2 layer MLP) 
Accuracy: 0.96 
F1: 0.9600298289705758 
Precision: 0.9602919457773134 
Recall: 0.96


In [8]:
# load data
data = pd.read_csv(DATA_PATH)

# split target labels from features
subset = X[['social_isolation_score', 
            'face_to_face_social_hours_weekly',
            'sleep_hours',
            'exercise_hours_weekly',
            'daily_gaming_hours',
            'academic_work_performance'
            ]]
y = data["gaming_addiction_risk_level"]

# pull out valiation set
X_full, X_val, y_full, y_val = train_test_split(X, y, test_size=0.2)

# perform train/test split
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=1)

# normalize datasets based on stats in train data
X_train_subset = scaler.fit_transform(X_train)
X_test_subset = scaler.transform(X_test)
X_val_subset = scaler.transform(X_val)

## MLP with one hidden layer
# run train/test loop
model_1L.fit(X_train_subset, y_train)

# test predictions
y_test_pred = model_1L.predict(X_test_subset)

# validation predictions
y_val_pred = model_1L.predict(X_val_subset)

print("VAL (std normalization, 1 layer MLP)",
    "\nAccuracy:", accuracy_score(y_val, y_val_pred),
    "\nF1:", f1_score(y_val, y_val_pred, average='weighted'),
    "\nPrecision:", precision_score(y_val, y_val_pred, average='weighted'),
    "\nRecall:", recall_score(y_val, y_val_pred, average='weighted'))

## MLP with two hidden layers
# run train/test loop
model_2L.fit(X_train_subset, y_train)

# test predictions
y_test_pred = model_2L.predict(X_test_subset)

# validation predictions
y_val_pred = model_2L.predict(X_val_subset)

print("VAL (std normalization, 2 layer MLP)",
    "\nAccuracy:", accuracy_score(y_val, y_val_pred),
    "\nF1:", f1_score(y_val, y_val_pred, average='weighted'),
    "\nPrecision:", precision_score(y_val, y_val_pred, average='weighted'),
    "\nRecall:", recall_score(y_val, y_val_pred, average='weighted'))


VAL (std normalization, 1 layer MLP) 
Accuracy: 0.93 
F1: 0.9304680300911439 
Precision: 0.9313095238095238 
Recall: 0.93
VAL (std normalization, 2 layer MLP) 
Accuracy: 0.955 
F1: 0.9551999999999999 
Precision: 0.9555227596017071 
Recall: 0.955
